In [0]:
%pip install kafka-python confluent-kafka websocket-client

In [0]:
%restart_python
dbutils.library.restartPython()

In [0]:
BOOTSTRAP_SERVERS = "pkc-oxqxx9.us-east-1.aws.confluent.cloud:9092"

TOPIC = "crypto-transaction"

API_KEY = "E7FIZIBKNYJXFO4K"
API_SECRET = "cflt1GuGMjLlEF431MS+O0gIcFHAxOgMA9p+KRV5j3IARQQ8cFFNPAtm9+yXkjbQ"

In [0]:
raw_stream_df = (
    spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS)
        .option("subscribe", TOPIC)

        .option("kafka.security.protocol", "SASL_SSL")
        .option("kafka.sasl.mechanism", "PLAIN")

        .option(
            "kafka.sasl.jaas.config",
            f"""
            kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required
            username="{API_KEY}"
            password="{API_SECRET}";
            """
        )

        .option("startingOffsets", "earliest")
        .load()
)

In [0]:
from pyspark.sql.functions import (col,hour,dayofmonth,current_date,to_date,current_timestamp)
bronze_df = raw_stream_df.select(
    col("key").cast("string").alias("key"),
    col("value").cast("string").alias("raw_json"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp")
).withColumn(
    "ingestion_date",
    current_date()
).withColumn(
    "hour",
    hour(col("timestamp"))
).withColumn(
    "day",
    dayofmonth(col("timestamp"))
)

In [0]:
bronze_query = (
    bronze_df.writeStream
        .format("delta")
        .outputMode("append")
        .partitionBy("day", "hour")
        .option(
            "checkpointLocation",
            "s3a://sebastian-crypto-lakehouse/checkpoints/bronze/crypto_transactions/"
        )
        .trigger(availableNow=True)#.trigger(processingTime="30 seconds")
        .start(
            "s3a://sebastian-crypto-lakehouse/bronze/crypto_transactions/"
        )
)

In [0]:
bronze_check_df = spark.read.format("delta").load(
    "s3a://sebastian-crypto-lakehouse/bronze/crypto_transactions/"
)

display(bronze_check_df)